In [1]:
import sqlite3
import numpy as np
import pandas as pd

con = sqlite3.connect("../banco/nice_rooms.db")
d = pd.read_sql("SELECT d.*, h.tier, h.name AS hotel FROM daily_market d JOIN hotels h USING (hotel_id)",
                con, parse_dates=["date"])
w = pd.read_sql("SELECT * FROM v_semanal", con, parse_dates=["week_start"])
r = pd.read_sql("SELECT r.*, h.tier FROM reviews r JOIN hotels h USING (hotel_id)", con)
d["evento"] = d["demand_event"].fillna("").ne("")

def agregar(df, por):
    """Ocupação e ADR ponderados (soma dividida pela soma), não média de médias."""
    s = df.groupby(por).agg(vendidos=("rooms_sold", "sum"), disponiveis=("rooms_available", "sum"),
                            receita=("room_revenue", "sum"))
    s["ocupacao"] = (s["vendidos"] / s["disponiveis"]).round(3)
    s["adr"] = (s["receita"] / s["vendidos"]).round(2)
    return s[["ocupacao", "adr"]]

In [2]:
oc = d.groupby("tier")["occupancy"].agg(media="mean", mediana="median", desvio="std", assimetria="skew").round(3)
oc

,media,mediana,desvio,assimetria
tier,,,,
boutique,0.758,0.750,0.104,0.197
economy,0.619,0.616,0.090,0.409
luxury,0.723,0.717,0.100,0.314
midscale,0.683,0.674,0.097,0.362
upscale,0.730,0.721,0.103,0.283


In [3]:
v = d.groupby("tier")["adr"].agg(media="mean", desvio="std")
v["cv_pct"] = (v["desvio"] / v["media"] * 100).round(1)
v.sort_values("cv_pct", ascending=False)

,media,desvio,cv_pct
tier,,,
boutique,211.047647,51.380888,24.3
upscale,223.514323,43.054781,19.3
luxury,367.233530,68.404702,18.6
economy,91.215234,16.559032,18.2
midscale,151.758864,27.678034,18.2


In [4]:
f = r["rating"].value_counts().sort_index().to_frame("absoluta")
f["relativa_%"] = (f["absoluta"] / f["absoluta"].sum() * 100).round(1)
f["acumulada_%"] = f["relativa_%"].cumsum().round(1)
print(f)
print("Notas 3 ou menos:", round((r["rating"] <= 3).mean() * 100, 1), "%")

        absoluta  relativa_%  acumulada_%
rating                                   
1           1087         9.0          9.0
2           2439        20.2         29.2
3           3212        26.6         55.8
4           2302        19.1         74.9
5           3037        25.1        100.0
Notas 3 ou menos: 55.8 %


In [5]:
ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(agregar(d, ["tier", "month"]).unstack(0)["ocupacao"])          # mês
print(agregar(d, ["tier", "weekday"]).unstack(0)["ocupacao"].reindex(ordem_dias))   # dia da semana
agregar(d, ["tier", "evento"])                                         # evento vs. noite comum

tier   boutique  economy  luxury  midscale  upscale
month                                              
1         0.649    0.529   0.619     0.584    0.624
2         0.671    0.551   0.641     0.603    0.647
3         0.711    0.582   0.680     0.639    0.685
4         0.752    0.617   0.720     0.678    0.727
5         0.785    0.645   0.750     0.709    0.761
6         0.838    0.689   0.801     0.757    0.811
7         0.868    0.718   0.833     0.785    0.843
8         0.895    0.745   0.863     0.814    0.868
9         0.815    0.665   0.775     0.733    0.784
10        0.758    0.619   0.726     0.681    0.731
11        0.674    0.553   0.647     0.609    0.650
12        0.664    0.542   0.635     0.597    0.640
tier       boutique  economy  luxury  midscale  upscale
weekday                                                
Monday        0.677    0.555   0.648     0.609    0.653
Tuesday       0.753    0.615   0.718     0.678    0.726
Wednesday     0.772    0.634   0.740     0.695  

ocupacao     adr
tier     evento                  
boutique False      0.751  217.32
         True       0.848  260.06
economy  False      0.616   92.96
         True       0.709  112.30
luxury   False      0.718  377.68
         True       0.821  455.75
midscale False      0.677  153.63
         True       0.778  186.08
upscale  False      0.725  225.67
         True       0.826  272.07

In [6]:
print("Ocupação × ADR (mercado todo):", round(d["occupancy"].corr(d["adr"]), 3))
print(pd.Series({t: g["occupancy"].corr(g["adr"]) for t, g in d.groupby("tier")}).round(3))

print("Nota × RevPAR (semanal, todos):", round(w["mean_rating"].corr(w["revpar"]), 3))
print(pd.Series({t: g["mean_rating"].corr(g["revpar"]) for t, g in w.groupby("tier")}).round(3))

notas = ["cleanliness", "comfort", "service_score", "location_score", "value_score", "sleep_quality"]
r[notas + ["rating"]].corr()["rating"].drop("rating").sort_values(ascending=False).round(3)

Ocupação × ADR (mercado todo): 0.603
boutique    0.722
economy     0.973
luxury      0.952
midscale    0.961
upscale     0.954
dtype: float64
Nota × RevPAR (semanal, todos): 0.22
boutique   -0.029
economy     0.031
luxury     -0.016
midscale   -0.056
upscale     0.032
dtype: float64


cleanliness       0.901
comfort           0.885
service_score     0.878
sleep_quality     0.867
value_score       0.809
location_score    0.166
Name: rating, dtype: float64

In [7]:
hotel = w["hotel_name"].iloc[0]            # troque pelo hotel que preferir
h = w[w["hotel_name"] == hotel].copy()
h["mes"] = h["week_start"].dt.to_period("M")
m = h.groupby("mes").agg(receita=("room_revenue", "sum"), disp=("available_rooms", "sum"))
m["revpar"] = m["receita"] / m["disp"]

x = np.arange(len(m))
a, b = np.polyfit(x, m["revpar"], 1)
prev_2029 = a * np.arange(len(m), len(m) + 12) + b
print(hotel, "| tendência por mês:", round(a, 2), "| RevPAR médio previsto 2029:", round(prev_2029.mean(), 2))

Alder & Quay Hotel | tendência por mês: 0.59 | RevPAR médio previsto 2029: 182.39


# ANOTACOES